In [1]:
import numpy as np
from collections import Counter
import regex as re
import tiktoken
from functorch.einops import rearrange
from sympy.solvers.ode.riccati import val_at_inf

PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""" # GPT-2 regex 

In [2]:
text = "low low low low low lower lower newest newest newest newest newest newest widest widest widest"

# Let's initialize the vocabulary 

vocabulary = {}
for i in range(256):
    key = bytes([i])
    value = i 
    vocabulary[key] = value 

# Let's do the pretokenization 

splitted_text = text.split(" ")
frequency_table = {}

for word in splitted_text:
    key = []
    encoding = word.encode('utf-8')
    for byte in encoding:
        key.append(bytes([byte]))
    key = tuple(key)
    value = frequency_table.get(key, 0)
    frequency_table[key] = value + 1

print(f"Frequency table at step 0 is ", frequency_table)
# Let's do the iterative merging 

# Implement the naive approach firts. Given the 
# frequency_table generate the pair_table and find max_pair

def find_max_pair(freq_table: dict) -> (tuple, dict):
    # We count all the pairs
    pair_table = {}

    for key in freq_table:
        length = len(key)
        for i in range(length - 1):
            pair_key = tuple((key[i], key[i + 1]))
            pair_value = pair_table.get(pair_key, 0)
            pair_table[pair_key] = pair_value + freq_table[key] # current value + occurence of the word where the pair is
    
    # Now we find the maximum pair 

    max_pair = None
    max_occurence = -1
    for pair in pair_table:
        if pair_table[pair] >= max_occurence:
            max_occurence = pair_table[pair]
            max_pair = pair

    return max_pair, pair_table

def update_freq_table(freq_table: dict, max_pair: tuple) -> dict:
    new_freq_table = {}

    for key in freq_table:
        value = freq_table[key]
        if max_pair[0] not in key or max_pair[1] not in key:
            new_freq_table[key] = value
        else:
            new_key = []
            i = 0
            while i < len(key) - 1:
                if key[i] == max_pair[0] and key[i + 1] == max_pair[1]:
                    new_key.append(max_pair[0] + max_pair[1])
                    i += 2
                else:
                    new_key.append(key[i])
                    i += 1
            if i == len(key) - 1:
                new_key.append(key[i])
            new_key_tuple = tuple(new_key)
            new_freq_table[new_key_tuple] = value

    return new_freq_table

N = 8

for i in range(N):
    # Find the max_pair
    curr_max_pair, pair_table = find_max_pair(frequency_table)

    # Add it to the dictionary
    vocabulary[curr_max_pair[0] + curr_max_pair[1]] = len(vocabulary)

    # Update the frequency_table
    frequency_table = update_freq_table(frequency_table, curr_max_pair)
    # print(f"The merged pair is {curr_max_pair}")
    print(f"Frequency table at step {i + 1} is ", frequency_table)

# print(vocabulary)

Frequency table at step 0 is  {(b'l', b'o', b'w'): 5, (b'l', b'o', b'w', b'e', b'r'): 2, (b'n', b'e', b'w', b'e', b's', b't'): 6, (b'w', b'i', b'd', b'e', b's', b't'): 3}
Frequency table at step 1 is  {(b'l', b'o', b'w'): 5, (b'l', b'o', b'w', b'e', b'r'): 2, (b'n', b'e', b'w', b'e', b'st'): 6, (b'w', b'i', b'd', b'e', b'st'): 3}
Frequency table at step 2 is  {(b'l', b'o', b'w'): 5, (b'l', b'o', b'w', b'e', b'r'): 2, (b'n', b'e', b'w', b'est'): 6, (b'w', b'i', b'd', b'est'): 3}
Frequency table at step 3 is  {(b'l', b'ow'): 5, (b'l', b'ow', b'e', b'r'): 2, (b'n', b'e', b'w', b'est'): 6, (b'w', b'i', b'd', b'est'): 3}
Frequency table at step 4 is  {(b'low',): 5, (b'low', b'e', b'r'): 2, (b'n', b'e', b'w', b'est'): 6, (b'w', b'i', b'd', b'est'): 3}
Frequency table at step 5 is  {(b'low',): 5, (b'low', b'e', b'r'): 2, (b'n', b'e', b'west'): 6, (b'w', b'i', b'd', b'est'): 3}
Frequency table at step 6 is  {(b'low',): 5, (b'low', b'e', b'r'): 2, (b'n', b'ewest'): 6, (b'w', b'i', b'd', b'est')

In [7]:
import regex as re
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

In [8]:
re.findall(PAT,"some text I'll pretokenize")

['some', ' text', ' I', "'ll", ' pretokenize']

In [43]:
from bpe_tokenizer_training import form_freq_table, form_pair_table

gpt2_splitted_text = re.findall(pattern=PAT, string=text)
freq_table = form_freq_table(input_text=gpt2_splitted_text)
# pair_table = form_pair_table(frequency_table=freq_table)

pair_table = Counter()
for key in freq_table:
    for i in range(len(key) - 1):
        pair_key = tuple((key[i], key[i + 1]))
        pair_table[pair_key] += freq_table[key]


In [44]:
freq_table

Counter({(b' ', b'n', b'e', b'w', b'e', b's', b't'): 6,
         (b' ', b'l', b'o', b'w'): 4,
         (b' ', b'w', b'i', b'd', b'e', b's', b't'): 3,
         (b' ', b'l', b'o', b'w', b'e', b'r'): 2,
         (b'l', b'o', b'w'): 1})

In [35]:
pair_table

Counter({(b'e', b's'): 9,
         (b's', b't'): 9,
         (b'w', b'e'): 8,
         (b'l', b'o'): 7,
         (b'o', b'w'): 7,
         (b' ', b'l'): 6,
         (b' ', b'n'): 6,
         (b'n', b'e'): 6,
         (b'e', b'w'): 6,
         (b' ', b'w'): 3,
         (b'w', b'i'): 3,
         (b'i', b'd'): 3,
         (b'd', b'e'): 3,
         (b'e', b'r'): 2})

In [31]:
test_string = "Hello"
test_string2 = "Hello "
print(list(test_string.encode('utf-8')))
print(list(test_string2.encode('utf-8')))

[72, 101, 108, 108, 111]
[72, 101, 108, 108, 111, 32]


In [39]:
print(bytes([0]).decode('utf-8'))

 


In [60]:
text = "hello how are you <|endoftext|>world"
pattern = "<|endoftext|>"
re.split(f'({pattern})', text)
# re.split(pattern, text)

['hello how are you ', '<', '|', 'endoftext', '|', '>', 'world']

In [66]:
test_string = "Héllò hôw <|endoftext|><|endoftext|> are ü? 🙃<|endoftext|>"

In [75]:
special_tokens = ["<|endoftext|>", "<|endoftext|><|endoftext|>"]
pattern = "|".join(re.escape(t) for t in sorted(special_tokens, reverse=True))
re.split(f'({pattern})', test_string)

['Héllò hôw ', '<|endoftext|><|endoftext|>', ' are ü? 🙃', '<|endoftext|>', '']

In [76]:
"asdabhsd".encode('utf-8')

b'asdabhsd'

In [77]:
pattern

'<\\|endoftext\\|><\\|endoftext\\|>|<\\|endoftext\\|>'

In [9]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")

In [10]:
import json
from pathlib import Path

vocab_path = Path("../tests/fixtures/gpt2_vocab.json")

with open(vocab_path, 'r') as file:
    vocab = json.load(file)

merges_path = Path("../tests/fixtures/gpt2_merges.txt")
merges = []
with open(merges_path, 'r') as file:
    for line in file:
        pair = re.split(pattern=' ', string= line.strip())
        merges.append(tuple((pair[0].encode('utf-8'), pair[1].encode('utf-8'))))

In [11]:
test_bytes = bytes([100, 100, 120])
print(test_bytes.decode('utf-8'))

ddx


In [12]:
merges[1]

(b'\xc4\xa0', b'a')

In [7]:
path = Path("../tests/fixtures/gpt2_vocab.json")
vocab = {}
with open(path, 'r') as file:
    inverse_vocab = json.load(file)     # maps from bytes to int

for byte_key in inverse_vocab:
    int_id = inverse_vocab[byte_key]
    byte_value = byte_key.encode('utf-8')
    vocab[int_id] = byte_value

In [8]:
vocab

{0: b'!',
 1: b'"',
 2: b'#',
 3: b'$',
 4: b'%',
 5: b'&',
 6: b"'",
 7: b'(',
 8: b')',
 9: b'*',
 10: b'+',
 11: b',',
 12: b'-',
 13: b'.',
 14: b'/',
 15: b'0',
 16: b'1',
 17: b'2',
 18: b'3',
 19: b'4',
 20: b'5',
 21: b'6',
 22: b'7',
 23: b'8',
 24: b'9',
 25: b':',
 26: b';',
 27: b'<',
 28: b'=',
 29: b'>',
 30: b'?',
 31: b'@',
 32: b'A',
 33: b'B',
 34: b'C',
 35: b'D',
 36: b'E',
 37: b'F',
 38: b'G',
 39: b'H',
 40: b'I',
 41: b'J',
 42: b'K',
 43: b'L',
 44: b'M',
 45: b'N',
 46: b'O',
 47: b'P',
 48: b'Q',
 49: b'R',
 50: b'S',
 51: b'T',
 52: b'U',
 53: b'V',
 54: b'W',
 55: b'X',
 56: b'Y',
 57: b'Z',
 58: b'[',
 59: b'\\',
 60: b']',
 61: b'^',
 62: b'_',
 63: b'`',
 64: b'a',
 65: b'b',
 66: b'c',
 67: b'd',
 68: b'e',
 69: b'f',
 70: b'g',
 71: b'h',
 72: b'i',
 73: b'j',
 74: b'k',
 75: b'l',
 76: b'm',
 77: b'n',
 78: b'o',
 79: b'p',
 80: b'q',
 81: b'r',
 82: b's',
 83: b't',
 84: b'u',
 85: b'v',
 86: b'w',
 87: b'x',
 88: b'y',
 89: b'z',
 90: b'{',
 91: b'|

In [9]:
b = bytes([128])   # one of the initial vocab entries
s = b.decode('latin-1')

In [179]:
text = ""
for i in range(5):
    print(f'the length of {i+1}th example is {len(dataset['train']['text'][i])}')
    text += dataset['train']['text'][i]
    text += '<|endoftext|>'
print(len(text))

with open("../cs336_basics/test.txt", 'w') as file:
    file.write(text)

the length of 1th example is 701
the length of 2th example is 705
the length of 3th example is 822
the length of 4th example is 845
the length of 5th example is 638
3776


In [178]:
len(dataset['train']['text'])

2119719

In [11]:
merge_list = [ tuple(('a'.encode('utf-8'), 'b'.encode('utf-8'))), 
              tuple(('ab'.encode('utf-8'), 'bc'.encode('utf-8')))
              ]
with open('../output/merges_list_test.txt', 'w') as file:
    for i in range(len(merge_list)):    # merge list is list[tuple[bytes, bytes]]
        first_token = merge_list[i][0].decode('utf-8')
        second_token = merge_list[i][1].decode('utf-8')
        file.write(first_token + " " + second_token + '\n')


In [12]:
vocab

{0: b'!',
 1: b'"',
 2: b'#',
 3: b'$',
 4: b'%',
 5: b'&',
 6: b"'",
 7: b'(',
 8: b')',
 9: b'*',
 10: b'+',
 11: b',',
 12: b'-',
 13: b'.',
 14: b'/',
 15: b'0',
 16: b'1',
 17: b'2',
 18: b'3',
 19: b'4',
 20: b'5',
 21: b'6',
 22: b'7',
 23: b'8',
 24: b'9',
 25: b':',
 26: b';',
 27: b'<',
 28: b'=',
 29: b'>',
 30: b'?',
 31: b'@',
 32: b'A',
 33: b'B',
 34: b'C',
 35: b'D',
 36: b'E',
 37: b'F',
 38: b'G',
 39: b'H',
 40: b'I',
 41: b'J',
 42: b'K',
 43: b'L',
 44: b'M',
 45: b'N',
 46: b'O',
 47: b'P',
 48: b'Q',
 49: b'R',
 50: b'S',
 51: b'T',
 52: b'U',
 53: b'V',
 54: b'W',
 55: b'X',
 56: b'Y',
 57: b'Z',
 58: b'[',
 59: b'\\',
 60: b']',
 61: b'^',
 62: b'_',
 63: b'`',
 64: b'a',
 65: b'b',
 66: b'c',
 67: b'd',
 68: b'e',
 69: b'f',
 70: b'g',
 71: b'h',
 72: b'i',
 73: b'j',
 74: b'k',
 75: b'l',
 76: b'm',
 77: b'n',
 78: b'o',
 79: b'p',
 80: b'q',
 81: b'r',
 82: b's',
 83: b't',
 84: b'u',
 85: b'v',
 86: b'w',
 87: b'x',
 88: b'y',
 89: b'z',
 90: b'{',
 91: b'|

In [5]:
# Now, let's download or tokenizer and do some experiments.
from tokenizer import Tokenizer
vocab_path = '../output/vocab_tiny_stories.json'
merges_path = '../output/merges_list.txt'
special_tokens = ['<|endoftext|>']
custom_tokenizer = Tokenizer.from_files(vocab_filepath=vocab_path, merges_filepath=merges_path, special_tokens=special_tokens)

In [6]:
string_to_encode = "Hello World! My name is Sanzhar"

ids = custom_tokenizer.encode(text=string_to_encode)
print(ids)

decoded_string = custom_tokenizer.decode(ids= ids)
print(decoded_string)

byte_list = []
for id in ids:
    byte_list.append(custom_tokenizer.vocab[id])

print(byte_list)

[1412, 520, 301, 323, 33, 3186, 1313, 406, 297, 299, 122, 104, 286]
Hello World! My name is Sanzhar
[b'Hello', b' W', b'or', b'ld', b'!', b' My', b' name', b' is', b' S', b'an', b'z', b'h', b'ar']


In [13]:
custom_tokenizer.vocab[9999]

b'<|endoftext|>'

## Experiments for 2.7

In [1]:
# Now, let's download or tokenizer and do some experiments.
from tokenizer import Tokenizer
from data_preprocessing import get_batch
vocab_path = '../output/vocab_tiny_stories.json'
merges_path = '../output/merges_list.txt'
special_tokens = ['<|endoftext|>']
custom_tokenizer = Tokenizer.from_files(vocab_filepath=vocab_path, merges_filepath=merges_path, special_tokens=special_tokens)

In [2]:
# We sample 10 documents from TinyStories Dataset and find the compression ratio.
# I think it should be around 5. 

with open('../output/TinyStories.txt', 'r') as file:
    text = file.read(100000)

In [5]:
ids = custom_tokenizer.encode(text=text)
print(f"The compression ratio is {len(text)/len(ids):.3f} bytes per id")

ids = gpt2_tokenizer.encode(text, allowed_special = {'<|endoftext|>'})
print(f"The compression ratio is {len(text)/len(ids):.3f} bytes per id")

100%|██████████| 251/251 [00:00<00:00, 2028.99it/s]

The compression ratio is 4.181 bytes per id
The compression ratio is 4.107 bytes per id


In [5]:
print(f"The compression ratio is {len(text)/len(ids):.3f} bytes per id")

The compression ratio is 4.329 bytes per id


In [8]:
# Let's tokenize the whole TinyStories DataSet

with open('../output/TinyStories.txt', 'r') as file:
    text = file.read()

ids = np.array(custom_tokenizer.encode(text=text), dtype=np.uint16)
np.save('../output/np_training_set.npy', ids)

100%|██████████| 4239437/4239437 [1:02:32<00:00, 1129.91it/s]


In [11]:
from language_model import transformer_lm, cross_entropy_loss
from data_preprocessing import get_batch
import numpy as np
import torch as torch
from einops import rearrange

list_ids = np.load('../output/np_training_set.npy', mmap_mode='r')
print(f"Number of tokens {len(list_ids)}")
print(f"Number of batches is {len(list_ids)/(1024 * 64)}")

Number of tokens 466762201
Number of batches is 7122.225967407227


In [2]:
model = transformer_lm(vocab_size=10000, d_model=8, num_layers=4,
                       num_heads=4, d_ff=4, max_seq_len=1024, rope_theta= np.pi/360,)

In [15]:
from datetime import datetime
timestamp_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")


'2026-05-07 22:25:42'

In [3]:
total_params = sum(p.numel() for p in model.parameters())

# for name, param in model.named_parameters():
    # print(name)
    # print(param.numel())


print(total_params)

161480


In [28]:
label, target = get_batch(dataset=list_ids, batch_size=8, context_length=1024, device='cpu')

In [52]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")    # For training on my Macbook
print(device)
model.to(device)

mps


transformer_lm(
  (token_embeddings): Embedding()
  (layers): ModuleList(
    (0-3): 4 x transformer_block(
      (attn): multihead_self_attention_with_RoPE(
        (q_proj): Linear()
        (k_proj): Linear()
        (v_proj): Linear()
        (output_proj): Linear()
        (rope): RoPE()
      )
      (ffn): Positionwise_feedforward(
        (w1): Linear()
        (w2): Linear()
        (w3): Linear()
      )
      (ln1): RMSNorm()
      (ln2): RMSNorm()
    )
  )
  (ln_final): RMSNorm()
  (lm_head): Linear()
)

In [7]:
out = model.forward(x=label)

In [32]:
out.shape

torch.Size([8, 1024, 10000])

In [78]:
out_rearrange = rearrange(out, "batch_size seq_len vocab_size -> (batch_size seq_len) vocab_size")
target_rearrange = rearrange(target, "batch_size seq_len -> (batch_size seq_len)")

In [80]:
out_rearrange = out_rearrange.to(device)
target_rearrange = target_rearrange.to(device)

In [81]:
cross_entropy_loss(inputs=out_rearrange, targets=target_rearrange.long())

tensor(9.2116, device='mps:0', grad_fn=<MeanBackward0>)

In [82]:
print(out_rearrange.device, out_rearrange.dtype)
print(target_rearrange.device, target_rearrange.dtype)

mps:0 torch.float32
mps:0 torch.uint16


In [83]:
target_rearrange.to(device)
target_rearrange.device

device(type='mps', index=0)

In [84]:
target_rearrange.device

device(type='mps', index=0)

In [88]:
x = torch.randn(5, 5)
y = torch.arange(x.shape[-2], device = 'mps')

In [89]:
y.device

device(type='mps', index=0)

In [92]:
torch.tensor([[0.5]]).squeeze()

tensor(0.5000)

In [93]:
torch.tensor([[0.5]]).squeeze(-1)

tensor([0.5000])

In [4]:
import tiktoken
from tokenizer import Tokenizer
gpt2_tokenizer = tiktoken.get_encoding('gpt2')
gpt2_tokenizer.encode("HELLO")

[13909, 3069, 46]

In [4]:
my_gpt2_tokenizer = Tokenizer.from_files(vocab_filepath='../tests/fixtures/gpt2_vocab.json',
                                         merges_filepath='../tests/fixtures/gpt2_merges.txt',
                                         special_tokens=['<|endoftext|>'])

In [5]:
# Now, let's download or tokenizer and do some experiments.
from tokenizer import Tokenizer
from data_preprocessing import get_batch
vocab_path = '../output/vocab_tiny_stories.json'
merges_path = '../output/merges_list.txt'
special_tokens = ['<|endoftext|>']
custom_tokenizer = Tokenizer.from_files(vocab_filepath=vocab_path, merges_filepath=merges_path, special_tokens=special_tokens)

In [6]:
my_gpt2_tokenizer.encode("HELLO")

100%|██████████| 1/1 [00:00<00:00, 346.26it/s]


[13909, 3069, 46]

In [31]:
custom_tokenizer.encode("HELLO")

100%|██████████| 1/1 [00:00<00:00, 16912.52it/s]


[8923, 76, 76, 79]

In [17]:
# FUCK I HAVE TO PRODUCE THE VALIDATION SET.
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")

In [49]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")

validation_ids = []
for i in range(len(ds['validation']['text'])):
    ids = custom_tokenizer.encode(ds['validation']['text'][i])
    validation_ids.extend(ids)
    validation_ids.extend([9999]) # endoftext token


100%|██████████| 1/1 [00:00<00:00, 653.22it/s]

In [56]:
# Now let's save the validation set
np.save('../output/np_validation_set.npy', validation_ids)

In [57]:
ids = custom_tokenizer.encode(text)
str_back = custom_tokenizer.decode(ids)

str_back == text

100%|██████████| 1/1 [00:00<00:00, 634.54it/s]


True